In [1]:
!unzip -q processed.zip -d data

import numpy as np
d = "data/processed/"
X_train = np.load(d + "X_train.npy")
X_test  = np.load(d + "X_test.npy")
y_train = np.load(d + "y_train.npy")
y_test  = np.load(d + "y_test.npy")
cat_train = np.load(d + "cat_train.npy")
cat_test  = np.load(d + "cat_test.npy")

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(145600, 46) (36400, 46) (145600,) (36400,)


In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [3]:
rf = RandomForestClassifier(n_estimators=100, criterion="gini", max_depth=None,
                            random_state=42, n_jobs=-1)

lr = LogisticRegression(penalty="l2", C=1.0, solver="lbfgs",
                        max_iter=1000, random_state=42)

In [4]:
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

In [5]:
def score(name, y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    print(name,
          "acc", round(accuracy_score(y_true, y_pred), 4),
          "prec", round(precision_score(y_true, y_pred), 4),
          "rec", round(recall_score(y_true, y_pred), 4),
          "f1", round(f1_score(y_true, y_pred), 4),
          "fpr", round(fp / (fp + tn), 4))

score("RF", y_test, rf_pred)
score("LR", y_test, lr_pred)

RF acc 0.9604 prec 0.9642 rec 0.9562 f1 0.9602 fpr 0.0355
LR acc 0.8479 prec 0.8909 rec 0.7928 f1 0.839 fpr 0.0971


In [6]:
for c in sorted(set(cat_test)):
    if c == "Benign":
        continue
    mask = cat_test == c
    print(c, "RF", round((rf_pred[mask] == 1).mean(), 3),
             "LR", round((lr_pred[mask] == 1).mean(), 3))

BruteForce RF 0.927 LR 0.58
DDoS RF 1.0 LR 0.992
DoS RF 1.0 LR 0.996
Mirai RF 1.0 LR 0.994
Recon RF 0.906 LR 0.708
Spoofing RF 0.891 LR 0.61
Web RF 0.97 LR 0.67


In [7]:
import pandas as pd

rows = []
for name, pred in [("RF", rf_pred), ("LR", lr_pred)]:
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    row = {"model": name, "condition": "clean",
           "accuracy": accuracy_score(y_test, pred),
           "precision": precision_score(y_test, pred),
           "recall": recall_score(y_test, pred),
           "f1": f1_score(y_test, pred),
           "fpr": fp / (fp + tn)}
    for c in sorted(set(cat_test)):
        if c != "Benign":
            row["recall_" + c] = (pred[cat_test == c] == 1).mean()
    rows.append(row)

df = pd.DataFrame(rows)
df.to_csv("baseline_results.csv", index=False)
df

,model,condition,accuracy,precision,recall,f1,fpr,recall_BruteForce,recall_DDoS,recall_DoS,recall_Mirai,recall_Recon,recall_Spoofing,recall_Web
0,RF,clean,0.960357,0.964209,0.956209,0.960192,0.035495,0.927308,1.000000,1.000000,1.000000,0.905769,0.890769,0.969615
1,LR,clean,0.847857,0.890899,0.792802,0.838993,0.097088,0.580385,0.991538,0.995769,0.993846,0.707692,0.610000,0.670385


In [8]:
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(n_neighbors=11, n_jobs=-1)
nn.fit(X_train)
dist, idx = nn.kneighbors(X_train)

print("self first:", (idx[:, 0] == np.arange(len(X_train))).mean())
neighbours = idx[:, 1:]
np.save("neighbours.npy", neighbours)
print(neighbours.shape)

self first: 0.9983173076923076
(145600, 10)


In [9]:
not_self = idx != np.arange(len(X_train))[:, None]
neighbours = idx[not_self].reshape(-1, 10)

print("self left:", (neighbours == np.arange(len(X_train))[:, None]).any())
np.save("neighbours.npy", neighbours)
print(neighbours.shape)

self left: False
(145600, 10)


In [10]:
def knn_flag(y_labels, neighbours, threshold=7):
    disagree = (y_labels[neighbours] != y_labels[:, None]).sum(axis=1)
    return disagree >= threshold

flagged = knn_flag(y_train, neighbours)
print("flagged on clean data:", flagged.sum())

import collections
print(collections.Counter(cat_train[flagged].tolist()))

flagged on clean data: 11585
Counter({'Benign': 3169, 'Spoofing': 2807, 'Recon': 2072, 'BruteForce': 2025, 'Web': 1502, 'DDoS': 5, 'Mirai': 4, 'DoS': 1})


In [11]:
def outlier_keep(X, y_labels, keep):
    keep = keep.copy()
    for cl in [0, 1]:
        rows = np.where(keep & (y_labels == cl))[0]
        d = np.linalg.norm(X[rows] - X[rows].mean(axis=0), axis=1)
        keep[rows[d > d.mean() + 3 * d.std()]] = False
    return keep

keep = outlier_keep(X_train, y_train, ~flagged)
print("kept:", keep.sum())
print("removed by outlier step:", (~flagged).sum() - keep.sum())

kept: 132729
removed by outlier step: 1286


In [12]:
rng = np.random.default_rng(42)
tests = [("random 10%", np.where(y_train == 1)[0], 0.10),
         ("DDoS 20%",   np.where(cat_train == "DDoS")[0], 0.20)]

for name, pool, rate in tests:
    flipped = rng.choice(pool, int(round(rate * len(pool))), replace=False)
    y_pois = y_train.copy()
    y_pois[flipped] = 0

    f = knn_flag(y_pois, neighbours)
    k = outlier_keep(X_train, y_pois, ~f)

    is_flip = np.zeros(len(y_train), bool)
    is_flip[flipped] = True
    removed = ~k
    print(name, "| flipped", len(flipped),
          "| caught", (removed & is_flip).sum(),
          "| clean rows removed", (removed & ~is_flip).sum())

random 10% | flipped 7280 | caught 5217 | clean rows removed 12188
DDoS 20% | flipped 2080 | caught 1873 | clean rows removed 12866
